# Módulo 2 — error longitudinal por zona

Error = (generada − fuente) − (destino real − fuente), con las medidas congeladas del módulo 1. Positivo = más envejecido que el destino real; MAE menor es mejor. Conserva los pares e imágenes guardados. No reajusta referencias ni ejecuta picking. Solo el bloque SD1.5 genera imágenes nuevas. Los errores incluyen captura/pose y no identifican causalmente sesgo demográfico.

In [ ]:
%pip install -e ".[aging-bias,evaluation]"


## 1. Rutas y evaluadores, una vez

Rutas del cuaderno de módulo 1 proporcionado. `REFERENCE_DIR` es de solo lectura; todo lo nuevo va a `OUTPUT`. DEX/KID no son necesarios para medir zonas. Edita `PICKING_DIR` si las variantes guardadas están en otra carpeta.

In [ ]:
from pathlib import Path
from dataclasses import replace
from types import SimpleNamespace
import pandas as pd
from src.aging_bias import FeatureConfig, PaperV2Config, WeightingConfig, SD15Config
from src.aging_bias.references import ReferenceBundle
from src.comparisong.evaluation_backends_v2 import prepare_paper_bundle
from src.comparisong.paper_analysis import run_paper_analysis
from src.comparisong.zone_paper import preflight_zone_protocol, display_zone_results

SAVED = Path("/content/drive/MyDrive/respaldo_test_comparison")
REFERENCE_DIR = Path("/content/drive/MyDrive/aging_bias_references_v2")
TEST_ROOT = Path("/content/drive/MyDrive/dataset_def/test")
CHECKPOINT_PATH = Path("/content/drive/MyDrive/adapter_inference.pt")
OUTPUT = Path("/content/drive/MyDrive/paper_comparison_zones_v2")
ASSETS = Path("/content/evaluation_models_v2")
PICKING_DIR = Path("/content/drive/MyDrive/paper_comparison_reduce_bias/ours_picking_variants")
MODELS = ["SAM", "FRAN", "FADING", "HRFAE", "ours_strength_0.31", "ours_picking_no_prompt", "ours_picking_prompt"]
references = ReferenceBundle.load(REFERENCE_DIR, load_training_data=False)
assert references.metadata.get("module1_close"), "Usa el bundle cerrado del módulo 1"
features = FeatureConfig(**references.metadata["config"]["features"])
picking_config = SimpleNamespace(output_dir=PICKING_DIR)  # Solo rutas; NO genera picking

config = PaperV2Config(
    test_manifest_path=SAVED / "prediction_manifest.csv", source_root=TEST_ROOT,
    generated_roots={**{m: SAVED / "generated" / m for m in MODELS},
        **{m: PICKING_DIR / "generated" / m for m in MODELS if m.startswith("ours_picking_")}},
    models=MODELS, reference_bundle_path=REFERENCE_DIR, output_dir=OUTPUT / "literature",
    features=features, weighting=WeightingConfig(min_bin_n=10),
    include_dex_appendix=False, include_kid=False, n_boot=1000, seed=2026,
    resume=True, batch_size=16, cache_dir=OUTPUT / "cache",
)

bundle = prepare_paper_bundle(ASSETS, feature_config=features, allow_download=True, include_dex=False, include_kid=False)


## 2. SD1.5 antes/después — generación pareada y métricas

Mismos pares de envejecimiento, semilla y strengths. Antes restaura la convolución original y desactiva adaptadores/condicionamiento de edad; después usa el checkpoint sin picking para aislar fine-tuning. `resume=True` reutiliza generaciones compatibles. El preflight prueba extracción real de zonas y archivos DINO antes de generar; no garantiza detectar rostros en todas las imágenes futuras. Este bloque es independiente del de literatura: puedes pasar al bloque 3 si solo quieres evaluar imágenes ya guardadas.

In [ ]:
from src.comparisong.sd15_counterfactual import run_sd15_before_after
validation_path = preflight_zone_protocol(config, bundle=bundle, assets_root=ASSETS, allow_download=True)
sd_config = SD15Config(
    checkpoint_path=CHECKPOINT_PATH, test_manifest_path=SAVED / "transitions.csv",
    source_root=TEST_ROOT, output_dir=OUTPUT / "sd15_counterfactual",
    validation_path=validation_path, anchors_reviewed=True,
    strengths=(0.10, 0.20, 0.30, 0.40, 0.50), seed=2026, resume=True,
    inference_config={"image_size":400, "num_inference_steps":34,
        "text_reference_mode":"source_age", "age_guidance_scale":3.0,
        "text_guidance_scale":7.0, "image_guidance_scale":1.5, "negative_prompt":""},
    local_files_only=False,
)
sd_outputs = run_sd15_before_after(sd_config)
with_sd = replace(config, test_manifest_path=sd_config.output_dir / "transitions.csv",
    models=list(sd_outputs["generated_roots"]), generated_roots=sd_outputs["generated_roots"],
    output_dir=sd_config.output_dir / "metrics")
sd_result = run_paper_analysis(with_sd, bundle=bundle, assets_root=ASSETS, allow_download=True, dino_device="auto")
display_zone_results(sd_result)


## 3. SAM, FRAN, FADING, HRFAE y variantes nuestras guardadas

La llamada pública es la misma. No genera imágenes ni repite picking. DINO carga la PCA congelada y usa GPU si hay CUDA; sus coordenadas son robustez latente, **no zonas anatómicas**. El reporte principal usa el soporte de edad validado por ICC del módulo 1; todas las edades se guardan también como exploratorias.

In [ ]:
result = run_paper_analysis(
    config, bundle=bundle, picking_config=picking_config,
    assets_root=ASSETS, allow_download=True, dino_device="auto",
)
display_zone_results(result)


## Salidas e interpretación

Cada bloque guarda `zone_errors_per_transition.csv`, `tables/zone_errors_aging.csv`, `tables/zone_errors_long_aging.csv`, `tables/component_errors_aging.csv`, `tables/ffhq_pattern_aging.csv`, diferencias pareadas, figuras y cobertura. SD1.5 añade `sd15_paired_zone_errors.csv`: **después − antes**, strength idéntico; MAE negativo indica mejora. Los pares son comunes entre modelos **por medida**, no se imputan fallos.

La hipótesis FFHQ se inspecciona en frente/surco/mandíbula: errores nativos negativos en arrugas y positivos en caída mandibular. Coincidir con esos signos es descriptivo, no prueba causal. `diagnostics/long_pair_support.csv` muestra cuántos pares largos reales hay; `long_pair_candidates.csv` lista transiciones adicionales de las mismas identidades. **No se incorporan ni generan automáticamente**: requieren generar todos los modelos sobre esos pares y definir una nueva evaluación.